# 01 · 先跑通一个完整 Loop

从上到下逐格执行。目标是跟踪一次 `task_list.py` 修复：**任务 → 请求 → 工具 → 观察 → 再请求 → 独立验证**。
本章使用最终代码 `code/mini_codex`，后面的 Python 类型、历史与分派器都来自它。
前半在本机研究消息；后半通过 Docker 内的 Linux/bubblewrap 真正读、改、运行候选。

运行环境：Python 3.12+；完整实验需要已启动的 Docker Desktop/Engine。缺少它会显示 `blocked`。
默认使用模型回放，工具和隔离仍真实运行；最后一个单元可手动开启 DeepSeek。
完整任务与局部机制实验分别记录结果；固定五个输入的通过不代表所有程序行为均已验证。


In [1]:
from pathlib import Path
import sys, json, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT / 'code'))
from mini_codex.learning import fresh_run_dir, show_source, run_chapter
from mini_codex.contracts import TaskInput, CallContext, ModelRequest, ToolCall, FixedPermit, data
from mini_codex.history import History
from mini_codex.model import ReplayModel
from mini_codex.dispatcher import Dispatcher, TOOL_SPECS
from mini_codex.loop import Loop, INSTRUCTIONS
EXPERIMENT = fresh_run_dir('ch01-message-flow')
print('本次消息实验目录：', EXPERIMENT)


本次消息实验目录： <project>/runs/notebooks/ch01-message-flow-20261113T124115Z-b5402cc4


## 1. 固定任务和判定，才能判断修复是否完成

缺陷把 `status` 错写成 `state`。控制器持有五个固定输入：两个应暴露缺陷，另三个用于防止修复破坏正常行为。
候选程序只接收输入并返回 JSON；断言由候选之外的验证器执行。先检查输入与原文，这一步不在本机执行候选。


In [2]:
checkpoint = ROOT / 'code'
source = (checkpoint / 'mini_codex/assets/fixtures/task-list/task_list.py').read_text()
cases = json.loads((checkpoint / 'mini_codex/assets/checks/cases.json').read_text())
assert 'task.get("state")' in source
assert len(cases) == 5 and sum(c['red'] for c in cases) == 2
print(source)
[(c['id'], c['input'], c['expected'], 'red见证' if c['red'] else '控制例') for c in cases]


"""JSON in / JSON out task-list program. Deliberate status-filter defect."""
import json
import sys


def filter_tasks(tasks, status=None):
    if status is None:
        return tasks
    return [task for task in tasks if task.get("state") == status]


if __name__ == "__main__":
    request = json.load(sys.stdin)
    print(json.dumps(filter_tasks(request["tasks"], request.get("status"))))



[('status-done',
  {'tasks': [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}],
   'status': 'done'},
  [{'id': 2, 'status': 'done'}],
  'red见证'),
 ('status-todo',
  {'tasks': [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}],
   'status': 'todo'},
  [{'id': 1, 'status': 'todo'}],
  'red见证'),
 ('all-control',
  {'tasks': [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}]},
  [{'id': 1, 'status': 'todo'}, {'id': 2, 'status': 'done'}],
  '控制例'),
 ('missing-control',
  {'tasks': [{'id': 1, 'status': 'todo'}], 'status': 'missing'},
  [],
  '控制例'),
 ('empty-control', {'tasks': [], 'status': 'done'}, [], '控制例')]

## 2. 一次采样只表达意图

`run_id` 关联任务，`sampling_id` 关联请求，模型的 `call_id` 关联工具及其结果。
改一改下面的任务文字，然后查看请求和响应。`ReplayModel` 只给出固定模型输出，此时没有执行工具。


In [3]:
task = TaskInput('TOPIC-01', '修复 status 筛选；保留 JSON stdin/stdout 接口。', '/workspace')
history = History('nb01', EXPERIMENT / 'events.jsonl')
history.append('task_input', data(task), [{'role': 'user', 'content': task.text}])
snapshot = history.snapshot()
context = CallContext('nb01', 'nb01:turn1', 'nb01:sample1', time.monotonic() + 300)
request = ModelRequest(context, snapshot.snapshot_id, INSTRUCTIONS,
                       snapshot.raw_items, TOOL_SPECS, {})
replay_model = ReplayModel()
response = replay_model.sample(request)
item = response.output_items[0]
call = ToolCall(item['call_id'], item['name'], json.loads(item['arguments']))
assert call.tool_name == 'search' and call.call_id == 'search-1'
print('task:', data(task))
print('sampling_id:', request.call_context.sampling_id)
print('模型提出的调用：', data(call))


task: {'task_id': 'TOPIC-01', 'text': '修复 status 筛选；保留 JSON stdin/stdout 接口。', 'workspace_ref': '/workspace'}
sampling_id: nb01:sample1
模型提出的调用： {'call_id': 'search-1', 'tool_name': 'search', 'arguments': {'query': 'status'}}


## 3. 历史要保留协议结构，快照不能成为旁路写入口

后续请求需要保留原始 `function_call`，再接同一 `call_id` 的 `function_call_output`。
这里先只追加模型意图，执行结果稍后由真实运行产生。尝试改快照，确认 History 没被改动。


In [4]:
history.append('model_items', {'sampling_id': context.sampling_id}, response.output_items)
before = history.snapshot()
editable_snapshot = history.snapshot()
editable_snapshot.raw_items[0]['content'] = '试图篡改快照'
after = history.snapshot()
assert before == after
assert after.raw_items[-1]['call_id'] == call.call_id
print('快照修改未影响 History；当前原始项数：', len(after.raw_items))
show_source(History.snapshot)


快照修改未影响 History；当前原始项数： 2


def snapshot(self):
        with self._lock:
            return HistorySnapshot(digest(self._items), tuple(copy.deepcopy(self._items)),
                                   self.revision, tuple(copy.deepcopy(self._sources)))

## 4. 工具意图必须先经过分派与许可

第一章固定四个工具。模型提出 `network` 并不会因此得到联网能力。
下例调用真实 Dispatcher，并把执行入口设成“触达即失败”的守卫，确认拒绝发生在执行之前；它不代表 OS 隔离测试。


In [5]:
class MustNotExecute:
    workspace = Path('/workspace')
    def execute(self, *args, **kwargs):
        raise AssertionError('未知工具不应该到达执行器')

invalid = ToolCall('denied-1', 'network', {'url': 'https://example.invalid'})
denied_context = CallContext('nb01', 'nb01:turn1', 'nb01:sample2',
                             time.monotonic() + 10, invalid.call_id)
dispatcher = Dispatcher(MustNotExecute(), history)
denied = dispatcher.dispatch(invalid, denied_context, FixedPermit('nb01', '/workspace'))
assert denied.outcome == 'rejected' and denied.error_code == 'unknown_or_denied_tool'
assert dispatcher.accepted == {}
data(denied)


{'call_id': 'denied-1',
 'outcome': 'rejected',
 'content': {},
 'side_effect_status': 'none',
 'cleanup_status': 'confirmed',
 'error_code': 'unknown_or_denied_tool'}

## 5. 终止是控制器的状态变更

有工具调用就执行、追加观察、再采样；只有文本结束时先独立验证，失败且有预算就继续。
`stop` 只提交一次，关闭新分派。之后冻结候选，再只读重验；“模型说完成”只是触发检查的输入。


In [6]:
show_source(Loop.stop)
# 改参数时，注意步数和绝对 deadline 是两个独立边界。
model_steps = len(ReplayModel().responses)
assert model_steps == 6
print('本章固定回放含', model_steps, '次采样；真实模型可以采用不同调用顺序。')


def stop(self, reason):
        # Engine.cancel uses this same lock. After this durable stop, only the
        # trusted final verifier/cleanup runs; a late cancel cannot re-open state.
        with self.lifecycle_lock:
            if self.terminal_reason is not None:
                return False
            # Cancellation may win the lock after candidate_check's earlier
            # observation but before this commit. Never publish readiness for a
            # cancellation already accepted by Engine.cancel. Unknown effects
            # keep their own stronger terminal reason.
            if reason == 'candidate_ready' and self.cancel.cancelled:
                reason = 'cancelled'
            self.terminal_reason = reason
            self.dispatcher.close()
            self.history.append("run_stopped", {"reason": reason})
            return True

本章固定回放含 6 次采样；真实模型可以采用不同调用顺序。


## 6. 先证明确实能隔离，再运行候选

执行同策略成对对照：工作区可读写；控制器能读的临时标记，worker 不能读；控制器可连的 loopback，worker 不能连。
Docker 外层是可信控制器；其运行权限用于启动嵌套 bubblewrap。候选代码仅在清空环境、受限挂载和独立 namespace 的 worker 内执行。


In [7]:
doctor = run_chapter('01', 'doctor')
sandbox_available = doctor['status'] != 'blocked'
if sandbox_available:
    preflight = json.loads((doctor['artifacts'] / 'preflight.json').read_text())
    assert preflight['status'] == 'passed'
    print({k: preflight[k] for k in ('status', 'platform', 'boundary')})
    print('workspace:', preflight['workspace_control']['output'])
else:
    print(doctor)


Running checkpoint 01/doctor; logs: <project>/runs/notebooks/checkpoint-01-doctor-20261113T124115Z-53d9220a


{'status': 'passed', 'platform': 'Linux-6.10.14-linuxkit-aarch64-with-glibc2.36', 'boundary': 'Linux controller container -> bubblewrap worker'}
workspace: allowed


## 7. 跑一次完整修复

`replay` 仅替换模型。搜索、读文件、补丁、shell、快照和验证都来自真实检查点。
每次运行使用新目录，重新从有缺陷的基线开始。


In [8]:
replay = run_chapter('01', 'replay') if sandbox_available else doctor
if replay['status'] != 'blocked':
    artifacts = replay['artifacts']
    baseline = json.loads((artifacts / 'baseline.json').read_text())
    final = json.loads((artifacts / 'final-verification.json').read_text())
    result = json.loads((artifacts / 'result.json').read_text())
    assert baseline['baseline_valid'] is True
    assert final['passed'] and final['assertion_count'] == final['expected_count'] == 5
    assert final['snapshot_unchanged'] and final['protection_intact']
    assert result['task_status'] == 'passed' and result['cleanup_status'] == 'confirmed'
    print('基线：', [(c['id'], c['passed']) for c in baseline['cases']])
    print('最终：', [(c['id'], c['passed']) for c in final['cases']])
    print((artifacts / 'candidate.diff').read_text())
else:
    print('完整修复 blocked：', replay.get('reason', replay.get('error')))


Running checkpoint 01/replay; logs: <project>/runs/notebooks/checkpoint-01-replay-20261113T124116Z-2a40e47a


基线： [('status-done', False), ('status-todo', False), ('all-control', True), ('missing-control', True), ('empty-control', True)]
最终： [('status-done', True), ('status-todo', True), ('all-control', True), ('missing-control', True), ('empty-control', True)]
--- baseline/task_list.py
+++ snapshot/task_list.py
@@ -6,7 +6,7 @@
 def filter_tasks(tasks, status=None):
     if status is None:
         return tasks
-    return [task for task in tasks if task.get("state") == status]
+    return [task for task in tasks if task.get("status") == status]
 
 
 if __name__ == "__main__":



## 8. 确认观察真正进入下一次模型请求

终端打印过工具输出，不等于模型已经收到。检查原始事件中的同一 `call_id`，再核对下一次请求的输入。


In [9]:
if replay['status'] != 'blocked':
    events = [json.loads(line) for line in (artifacts / 'events.jsonl').read_text().splitlines()]
    matched = []
    for index, event in enumerate(events):
        if event['kind'] != 'tool_observation':
            continue
        call_id = event['payload']['call_id']
        next_request = next(e for e in events[index + 1:] if e['kind'] == 'model_request')
        inputs = next_request['payload']['messages']
        assert any(i.get('type') == 'function_call_output' and i.get('call_id') == call_id for i in inputs)
        assert any(i.get('type') == 'function_call' and i.get('call_id') == call_id for i in inputs)
        matched.append((call_id, next_request['payload']['call_context']['sampling_id']))
    assert sum(e['kind'] == 'run_stopped' for e in events) == 1
    print('调用 → 下一次采样：')
    for row in matched:
        print(row)
else:
    print('尚无真实工具事件；先完成上一格的 Linux 运行条件。')


调用 → 下一次采样：
('search-1', 'run-b4c8493e096541faaceb02dc1c361171:sample2')
('read-1', 'run-b4c8493e096541faaceb02dc1c361171:sample3')
('test-red', 'run-b4c8493e096541faaceb02dc1c361171:sample4')
('patch-1', 'run-b4c8493e096541faaceb02dc1c361171:sample5')
('test-green', 'run-b4c8493e096541faaceb02dc1c361171:sample6')


## 9. 看失败实验，避免把“退出 0”当作修复成功

运行章内测试，包括假装输出 passed、候选直接 exit(0)、超时子进程、写后丢失回执、观察后取消。
下面读取的是本次真正产生的探针结果，不是预填成功值。


In [10]:
tests = run_chapter('01', 'test') if sandbox_available else doctor
if tests['status'] != 'blocked':
    test_artifacts = tests['artifacts']
    test_result = json.loads((test_artifacts / 'tests.json').read_text())
    assert test_result['failures'] == test_result['errors'] == test_result['skipped'] == 0
    print(test_result)
    for name in ('test_write_then_lost_confirmation_is_unknown_and_not_retried',
                 'test_cancel_after_observation_preserves_result_and_stops_new_work'):
        observed = json.loads((test_artifacts / 'probes' / name / 'result.json').read_text())
        print(name, {k: observed[k] for k in ('task_status', 'stop_reason', 'cleanup_status')})
else:
    print(tests)


Running checkpoint 01/test; logs: <project>/runs/notebooks/checkpoint-01-test-20261113T124119Z-40ac4c37


{'tests_run': 70, 'failures': 0, 'errors': 0, 'skipped': 0, 'scope': 'final_package', 'mode': 'replay', 'isolation': 'real LinuxWorker in integration tests; other tests label their ports'}
test_write_then_lost_confirmation_is_unknown_and_not_retried {'task_status': 'unknown', 'stop_reason': 'unknown_action', 'cleanup_status': 'confirmed'}
test_cancel_after_observation_preserves_result_and_stops_new_work {'task_status': 'cancelled', 'stop_reason': 'cancelled', 'cleanup_status': 'confirmed'}


## 10. 换成真实 DeepSeek

把`code/.env.example` 复制成私有 `.env.local`，设置密钥并将权限设为 600；不要把密钥填进 Notebook。
模型与服务地址必须显式配置。将下一格 `RUN_LIVE` 改为 `True` 才会请求付费服务。
每次请求限制输出 2048 tokens，任务还有步数和绝对时限。这里的成功仍只覆盖固定五个输入。


In [11]:
RUN_LIVE = False
if RUN_LIVE:
    live = run_chapter('01', 'live', env_file=ROOT / '.env.local')
    if live['status'] != 'blocked':
        live_result = json.loads((live['artifacts'] / 'result.json').read_text())
        assert live_result['mode'] == 'live' and live_result['task_status'] == 'passed'
        print(live_result)
    else:
        print(live)
else:
    print('本格未调用模型服务。设 RUN_LIVE=True 后再执行。')


本格未调用模型服务。设 RUN_LIVE=True 后再执行。


## 对照 Codex，再继续下一章

教学代码是 Python 同步薄实现；Codex 的相关控制链依据固定提交：

- [submission_loop：接收提交与其他事件](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/handlers.rs#L474-L538)
- [run_turn：判断继续采样](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn.rs#L663-L690)
- [采样时收集完成工具项](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn.rs#L3326-L3342)

Python 的完整响应后串行执行、固定许可与五例验证器属于教学选择。读取上述 Rust 测试和代码不算执行了它们。
下一章深入上下文：原始历史保留了什么，有限窗口又应让模型看到什么？
